# CMRReportAnalyzer: quickstart

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NedaTavakoli/CMRReportAnalyzer/blob/main/notebooks/quickstart.ipynb)

Classify myocardial scar etiology (**ischemic, non-ischemic, mixed, no-scar, not-specified**) from free-text cardiac MRI reports, as in the paper *Can AI Identify Myocardial Scar Etiology from Cardiac MRI Reports?*

1. **Rule-based NLP:** runs anywhere, no GPU, with the report shown in color.
2. **Your own report.**
3. **Zero-shot LLM (optional):** a public open-weight model on your own computer through [Ollama](https://ollama.com). No data leave the machine.
4. **Agreement check:** accept reports where the rules and the LLM agree; send the rest to an expert.
5. **Few-shot (optional):** add a few labeled example reports to the prompt.
6. **A CSV of reports.**

All reports in this notebook are **synthetic**. Research use only. Not for clinical use.

In [1]:
# Setup: in Google Colab this downloads the repository; locally, run the notebook from the repository.
import os, sys, subprocess
if "google.colab" in sys.modules and not os.path.exists("cmrreportanalyzer"):
    subprocess.run(["git", "clone", "-q", "https://github.com/NedaTavakoli/CMRReportAnalyzer.git", "cmrreportanalyzer"], check=True)
    os.chdir("cmrreportanalyzer")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "notebooks/requirements-notebook.txt"], check=True)
elif os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

import glob
import pandas as pd
from cmrreportanalyzer import classify_scar, extract_lvef, explain, show_report, classify_llm, ollama_available, compare, analyze_csv
print("ready")

ready


## 1. Rule-based NLP

The paper's rules: a fixed hierarchy (no LGE → no-scar; ischemic and non-ischemic evidence → mixed; then ischemic, non-ischemic, not-specified). In the paper they scored 90.2% against expert readers and classified 37,869 reports in under 4 minutes on a CPU. The rules were written by a cardiologist and tested against two radiologists.

In [2]:
reports = {os.path.basename(f)[:-4]: open(f).read() for f in sorted(glob.glob("examples/synthetic_reports/*.txt"))}
rows = []
for name, text in reports.items():
    e = explain(text)
    rows.append({"report": name, "class (rules)": e["label"], "LVEF %": extract_lvef(text), "rule matched": e["reason"] or "explicit no-LGE / scar statement"})
pd.DataFrame(rows)

,report,class (rules),LVEF %,rule matched
0,ischemic,ischemic,41.0,subendocardial (ischemic)
1,mixed,mixed,32.0,mixed (ischemic + non-ischemic)
2,no-scar,no-scar,62.0,explicit no-LGE / scar statement
3,no-scar_negated-wording,no-scar,59.0,explicit no-LGE / scar statement
4,non-ischemic,non-ischemic,50.0,mid-wall (non-ischemic)
5,not-specified,not-specified,48.0,explicit no-LGE / scar statement


### See the report in color

`show_report` shows the rule class and highlights the terms the rules look for: **anatomy** (yellow), **LGE pattern** (blue), **disease** (orange) and **no-LGE statements** (green). Grey text is a History or Technique section that the pattern rules skip. A highlighted term does not always decide the class.

In [3]:
show_report(reports["mixed"])

In [4]:
show_report(reports["no-scar_negated-wording"])

## 2. Your own report

Replace the text below with any LGE cardiac MRI report. Keep real patient reports on computers approved for them.

In [5]:
my_report = """
FINDINGS:
Left ventricle: Mildly dilated. LVEF 45%.
Late gadolinium enhancement: Subendocardial delayed enhancement of the basal inferior wall in the RCA territory.
IMPRESSION: Prior inferior myocardial infarction.
"""
explain(my_report)

{'label': 'ischemic',
 'reason': 'subendocardial (ischemic)',
 'ischemic_evidence': True,
 'nonischemic_evidence': False,
 'scar_mentioned': False}

In [6]:
show_report(my_report)

## 3. Zero-shot LLM (optional)

The paper ran public open-weight LLMs with no training: Qwen3-32B (89.8%) and Llama-3.1-70B (88.3%) matched the rules; Llama-3.1-8B reached 79.2%. You need [Ollama](https://ollama.com) and a model:

- **Laptop:** `ollama pull llama3.1:8b` (about 5 GB).
- **GPU with ≥24 GB:** `ollama pull qwen3:32b` (about 20 GB), the paper's best model.
- **Google Colab:** set `INSTALL_OLLAMA_IN_COLAB = True` below (a free T4 GPU runs `llama3.1:8b`).

The prompt and settings are the paper's: temperature 0, 8,192-token context, the model's own answer (no keyword rule), and an unreadable answer is returned as missing.

In [7]:
MODEL = "llama3.1:8b"            # or "qwen3:32b", "llama3.1:70b"
INSTALL_OLLAMA_IN_COLAB = False

if INSTALL_OLLAMA_IN_COLAB and "google.colab" in sys.modules:
    import time
    subprocess.run("curl -fsSL https://ollama.com/install.sh | sh", shell=True, check=True)
    subprocess.Popen(["ollama", "serve"], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL); time.sleep(5)
    subprocess.run(["ollama", "pull", MODEL], check=True)

available = ollama_available()
LLM_ON = MODEL in available
print("Ollama models:", available or "no Ollama server found", "| using", MODEL if LLM_ON else "rules only")

Ollama models: ['qwen3:32b', 'llama3.1:8b', 'llama3.1:70b'] | using llama3.1:8b


## 4. Agreement check

In the paper, where the rules and zero-shot Qwen3-32B agreed (85.6% of expert-labeled reports), 97.8% were correct, and the disagreements held most rule errors. So: **agree → accept; disagree → expert review.**

In [8]:
rows = []
for name, text in reports.items():
    rules = classify_scar(text)
    llm = classify_llm(text, model=MODEL) if LLM_ON else None
    rows.append({"report": name, "rules": rules, "zero-shot LLM": llm, "decision": compare(rules, llm)})
pd.DataFrame(rows)

,report,rules,zero-shot LLM,decision
0,ischemic,ischemic,ischemic,agree
1,mixed,mixed,mixed,agree
2,no-scar,no-scar,no-scar,agree
3,no-scar_negated-wording,no-scar,no-scar,agree
4,non-ischemic,non-ischemic,non-ischemic,agree
5,not-specified,not-specified,non-ischemic,review


## 5. Few-shot (optional)

Few-shot adds five labeled example reports to each prompt, chosen by similarity (one per class first). In the paper, five examples from a pool of as few as 10 expert-labeled reports raised Qwen3-32B to 95.3% and Llama-3.1-70B to 90.6%, but did not help smaller models.

Use **your own labeled reports** as the pool: a DataFrame with columns `report_text` and `gold_label`. Here, as an illustration only, each synthetic report is classified with the other synthetic reports as the pool. The not-specified report is the only one of its class, so when it is left out the pool has no not-specified example: **the example pool should include every class.** In the paper the pool held 194 expert-labeled reports.

In [9]:
if LLM_ON:
    rows = []
    for name, text in reports.items():
        pool = pd.DataFrame({"report_text": [t for n, t in reports.items() if n != name],
                             "gold_label": [classify_scar(t) for n, t in reports.items() if n != name]})
        rows.append({"report": name, "rules": classify_scar(text), "few-shot LLM": classify_llm(text, model=MODEL, examples=pool)})
    display(pd.DataFrame(rows))
else:
    print("Start Ollama with a model (section 3) to run few-shot.")

,report,rules,few-shot LLM
0,ischemic,ischemic,ischemic
1,mixed,mixed,mixed
2,no-scar,no-scar,no-scar
3,no-scar_negated-wording,no-scar,no-scar
4,non-ischemic,non-ischemic,non-ischemic
5,not-specified,not-specified,ischemic


## 6. A CSV of reports

`analyze_csv` adds `scar_rules` and `lvef`, and with a model also `scar_llm` and `decision`.

In [10]:
pd.DataFrame({"id": list(reports), "report_text": list(reports.values())}).to_csv("example_reports.csv", index=False)
out = analyze_csv("example_reports.csv", "report_text", model=MODEL if LLM_ON else None)
os.remove("example_reports.csv")
out.drop(columns="report_text")

,id,scar_rules,lvef,scar_llm,decision
0,ischemic,ischemic,41.0,ischemic,agree
1,mixed,mixed,32.0,mixed,agree
2,no-scar,no-scar,62.0,no-scar,agree
3,no-scar_negated-wording,no-scar,59.0,no-scar,agree
4,non-ischemic,non-ischemic,50.0,non-ischemic,agree
5,not-specified,not-specified,48.0,non-ischemic,review


## Notes

- The LLMs are public models downloaded by you; nothing in this repository was trained on patient data, and no patient data are included.
- Deployment settings matter: set the context window explicitly and record failed answers as missing (see the paper).
- Code and paper: https://github.com/NedaTavakoli/CMRReportAnalyzer. Please cite the paper if you use this code.